# 09 — Pipeline Chaining & Multi-Validator Guards

32 examples covering .use(), .use_many(), validator execution order, mixed OnFailActions,
input vs output guards, dynamic pipelines, async pipelines, and real-world guard stacks.

**Installation:**
```bash
pip install guardrails-ai openai python-dotenv
guardrails hub install hub://guardrails/toxic_language
guardrails hub install hub://guardrails/profanity_free
guardrails hub install hub://guardrails/valid_length
guardrails hub install hub://guardrails/detect_pii
guardrails hub install hub://guardrails/reading_level
guardrails hub install hub://guardrails/gibberish_text
guardrails hub install hub://guardrails/competitor_check
guardrails hub install hub://guardrails/restrict_to_topic
guardrails hub install hub://guardrails/contains_string
```

In [ ]:
import os, time, asyncio
from dotenv import load_dotenv
load_dotenv('../.env')

import openai
from guardrails import Guard, AsyncGuard, OnFailAction
from guardrails.errors import ValidationError

oai = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = 'gpt-4o-mini'
print('Setup complete.')

In [ ]:
!guardrails hub install hub://guardrails/toxic_language --quiet
!guardrails hub install hub://guardrails/profanity_free --quiet
!guardrails hub install hub://guardrails/valid_length --quiet
!guardrails hub install hub://guardrails/detect_pii --quiet
!guardrails hub install hub://guardrails/reading_level --quiet
!guardrails hub install hub://guardrails/gibberish_text --quiet
!guardrails hub install hub://guardrails/competitor_check --quiet
!guardrails hub install hub://guardrails/restrict_to_topic --quiet
!guardrails hub install hub://guardrails/contains_string --quiet

## Examples 01–04: .use() and .use_many() Basics

In [ ]:
# Example 01: .use() with a single validator
from guardrails.hub import ValidLength
guard = Guard().use(ValidLength(min=10, max=500, on_fail=OnFailAction.EXCEPTION))
outcome = guard.validate('This is a test string that is long enough.')
print('single .use() passed:', outcome.validation_passed)

In [ ]:
# Example 02: Chained .use().use() — two validators applied in sequence
from guardrails.hub import ValidLength, ToxicLanguage
guard = (
    Guard()
    .use(ValidLength(min=10, max=500, on_fail=OnFailAction.EXCEPTION))
    .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
)
outcome = guard.validate('This is a professional and informative response about software development.')
print('chained .use() passed:', outcome.validation_passed)

In [ ]:
# Example 03: .use_many() with a list of validators
from guardrails.hub import ValidLength, ToxicLanguage, ProfanityFree
validators = [
    ValidLength(min=10, max=500, on_fail=OnFailAction.EXCEPTION),
    ToxicLanguage(on_fail=OnFailAction.EXCEPTION),
    ProfanityFree(on_fail=OnFailAction.EXCEPTION),
]
guard = Guard().use_many(*validators)
outcome = guard.validate('Our platform delivers reliable enterprise software solutions.')
print('.use_many() passed:', outcome.validation_passed)

In [ ]:
# Example 04: .use_many() vs chained .use() — identical behavior, different syntax
from guardrails.hub import ValidLength, ToxicLanguage
guard_chained = (
    Guard()
    .use(ValidLength(min=5, max=200, on_fail=OnFailAction.NOOP))
    .use(ToxicLanguage(on_fail=OnFailAction.NOOP))
)
guard_many = Guard().use_many(
    ValidLength(min=5, max=200, on_fail=OnFailAction.NOOP),
    ToxicLanguage(on_fail=OnFailAction.NOOP)
)
text = 'A professional message about engineering.'
r1 = guard_chained.validate(text)
r2 = guard_many.validate(text)
print('chained result  :', r1.validation_passed)
print('use_many result :', r2.validation_passed)
print('identical?', r1.validation_passed == r2.validation_passed)

## Examples 05–08: Execution Order and OnFailAction Interactions

In [ ]:
# Example 05: Validator execution order — first validator runs first
from guardrails.hub import ValidLength, ToxicLanguage

# ValidLength runs first; if it throws EXCEPTION, ToxicLanguage never executes
guard = (
    Guard()
    .use(ValidLength(min=50, max=500, on_fail=OnFailAction.EXCEPTION))  # runs 1st
    .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))                  # runs 2nd
)
try:
    guard.validate('Short.')  # fails at ValidLength, ToxicLanguage never checked
except ValidationError as e:
    print('First validator raised, second skipped. Error:', str(e)[:60])

In [ ]:
# Example 06: Short-circuit with EXCEPTION — second validator never runs
from guardrails.hub import ValidLength, GibberishText
guard = (
    Guard()
    .use(ValidLength(min=100, max=1000, on_fail=OnFailAction.EXCEPTION))
    .use(GibberishText(threshold=0.5, on_fail=OnFailAction.EXCEPTION))
)
try:
    guard.validate('asjdh')  # too short AND gibberish — only first error raised
except ValidationError as e:
    print('Short-circuit: only ValidLength error:', 'ValidLength' in str(e) or 'length' in str(e).lower())

In [ ]:
# Example 07: All validators run when using NOOP — collects all failures
from guardrails.hub import ValidLength, GibberishText
guard = (
    Guard()
    .use(ValidLength(min=100, max=1000, on_fail=OnFailAction.NOOP))  # NOOP: doesn't stop pipeline
    .use(GibberishText(threshold=0.5, on_fail=OnFailAction.NOOP))    # also runs
)
outcome = guard.validate('asjdh')  # too short AND gibberish
print('Both validators ran (NOOP). validation_passed:', outcome.validation_passed)
if guard.history:
    logs = guard.history[0].iterations[0].validator_logs
    for log in logs:
        print(f'  {log.validator_name}: {type(log.validation_result).__name__}')

In [ ]:
# Example 08: Mixed OnFailActions in same pipeline
from guardrails.hub import ValidLength, ToxicLanguage, ProfanityFree
guard = (
    Guard()
    .use(ValidLength(min=5, max=1000, on_fail=OnFailAction.FIX))       # FIX: corrects, keeps going
    .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))                  # EXCEPTION: stops pipeline
    .use(ProfanityFree(on_fail=OnFailAction.NOOP))                       # only reached if toxic passes
)
clean = 'Our team is dedicated to delivering excellent products.'
outcome = guard.validate(clean)
print('mixed actions — clean text passed:', outcome.validation_passed)

## Examples 09–11: Input vs Output Guards, Field-Specific

In [ ]:
# Example 09: Input guard (validates user prompt) vs output guard (validates LLM response)
from guardrails.hub import ValidLength, DetectPII

# Input guard — checks user input before sending to LLM
input_guard = Guard().use(ValidLength(min=5, max=500, on_fail=OnFailAction.EXCEPTION))

# Output guard — checks LLM response
output_guard = Guard().use(DetectPII(on_fail=OnFailAction.EXCEPTION))

user_input = 'What is the capital of Australia?'
input_guard.validate(user_input)  # validate input first

llm_resp = oai.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': user_input}]
)
llm_text = llm_resp.choices[0].message.content

outcome = output_guard.validate(llm_text)  # validate output after
print('input+output dual guard passed:', outcome.validation_passed)

In [ ]:
# Example 10: Field-specific validator application on Pydantic model
from pydantic import BaseModel
from guardrails.hub import ToxicLanguage, ValidLength

class BlogPost(BaseModel):
    title: str
    body: str

guard = (
    Guard.for_pydantic(output_class=BlogPost)
    .use(ValidLength(min=5, max=200, on_fail=OnFailAction.EXCEPTION))
    .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
)
outcome = guard.validate('{"title": "Python Tips", "body": "Here are some tips for writing clean Python code."}')
print('blog post validated:', outcome.validated_output)

In [ ]:
# Example 11: Two-stage pipeline — input guard then output guard in sequence
from guardrails.hub import ValidLength, ToxicLanguage

def safe_llm_call(user_input: str) -> str:
    # Stage 1: Validate input
    input_guard = Guard().use(ValidLength(min=5, max=500, on_fail=OnFailAction.EXCEPTION))
    input_guard.validate(user_input)

    # Stage 2: Call LLM
    resp = oai.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': user_input}]
    )
    llm_text = resp.choices[0].message.content

    # Stage 3: Validate output
    output_guard = Guard().use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
    output = output_guard.validate(llm_text)
    return output.validated_output

result = safe_llm_call('Explain what REST means in web development.')
print('two-stage pipeline result:', result[:100] if result else 'None')

## Examples 12–15: Domain-Specific Pipelines

In [ ]:
# Example 12: 3-validator safety stack — ToxicLanguage + ProfanityFree + DetectPII
from guardrails.hub import ToxicLanguage, ProfanityFree, DetectPII
safety_guard = (
    Guard()
    .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
    .use(ProfanityFree(on_fail=OnFailAction.EXCEPTION))
    .use(DetectPII(on_fail=OnFailAction.EXCEPTION))
)
outcome = safety_guard.validate('Our team is excited to announce a new feature update.')
print('3-validator safety stack passed:', outcome.validation_passed)

In [ ]:
# Example 13: Content quality pipeline — GibberishText + ReadingLevel + ValidLength
from guardrails.hub import GibberishText, ReadingLevel, ValidLength
quality_guard = (
    Guard()
    .use(GibberishText(threshold=0.5, on_fail=OnFailAction.EXCEPTION))
    .use(ReadingLevel(reading_level=10, on_fail=OnFailAction.NOOP))
    .use(ValidLength(min=30, max=2000, on_fail=OnFailAction.EXCEPTION))
)
outcome = quality_guard(
    oai.chat.completions.create,
    prompt='Explain what machine learning is in a paragraph.',
    model=MODEL
)
print('quality pipeline passed:', outcome.validation_passed)

In [ ]:
# Example 14: Brand safety pipeline — CompetitorCheck + ContainsString
from guardrails.hub import CompetitorCheck, ContainsString
brand_guard = (
    Guard()
    .use(CompetitorCheck(competitors=['CompetitorCorp', 'RivalTech'], on_fail=OnFailAction.EXCEPTION))
    .use(ContainsString(search_string='Acme', on_fail=OnFailAction.EXCEPTION))  # must mention own brand
)
try:
    brand_guard.validate('Acme offers superior products compared to CompetitorCorp.')
except ValidationError:
    print('FAIL - competitor name in brand response')

outcome = brand_guard.validate('Acme delivers enterprise-grade reliability at every scale.')
print('PASS - clean brand message:', outcome.validation_passed)

In [ ]:
# Example 15: Customer support pipeline — ProfanityFree + ValidLength + ContainsString
from guardrails.hub import ProfanityFree, ValidLength, ContainsString
support_guard = (
    Guard()
    .use(ProfanityFree(on_fail=OnFailAction.EXCEPTION))
    .use(ValidLength(min=20, max=1000, on_fail=OnFailAction.EXCEPTION))
    .use(ContainsString(search_string='contact', on_fail=OnFailAction.NOOP))  # prefer including contact info
)
outcome = support_guard(
    oai.chat.completions.create,
    prompt='Write a customer support response for a billing issue. Mention they can contact support.',
    model=MODEL
)
print('support pipeline passed:', outcome.validation_passed)

## Examples 16–21: Advanced Pipeline Patterns

In [ ]:
# Example 16: RAG pipeline — ValidLength + ContainsString (requires source citation)
from guardrails.hub import ValidLength, ContainsString
rag_guard = (
    Guard()
    .use(ValidLength(min=50, max=2000, on_fail=OnFailAction.EXCEPTION))
    .use(ContainsString(search_string='according to', on_fail=OnFailAction.NOOP))  # should cite source
)
outcome = rag_guard(
    oai.chat.completions.create,
    prompt='Summarize what guardrails-ai does. Cite your source with "According to" at the start.',
    model=MODEL
)
print('RAG pipeline passed:', outcome.validation_passed)

In [ ]:
# Example 17: Conditional validator — choose based on user role
from guardrails.hub import ValidLength, ToxicLanguage

def get_guard_for_role(role: str) -> Guard:
    base = Guard().use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
    if role == 'admin':
        # Admins get looser length constraints
        return base.use(ValidLength(min=5, max=5000, on_fail=OnFailAction.NOOP))
    else:
        # Regular users get stricter length
        return base.use(ValidLength(min=20, max=500, on_fail=OnFailAction.EXCEPTION))

for role in ['admin', 'user']:
    guard = get_guard_for_role(role)
    outcome = guard.validate('This response is well-formed and professional content for the platform.')
    print(f'  role={role}: passed={outcome.validation_passed}')

In [ ]:
# Example 18: Error aggregation across all validators before raising
from guardrails.hub import ValidLength, ToxicLanguage, ProfanityFree
guard = (
    Guard()
    .use(ValidLength(min=50, max=500, on_fail=OnFailAction.NOOP))
    .use(ToxicLanguage(on_fail=OnFailAction.NOOP))
    .use(ProfanityFree(on_fail=OnFailAction.NOOP))
)
outcome = guard.validate('Hi!')  # short AND potentially problematic
if not outcome.validation_passed and guard.history:
    logs = guard.history[0].iterations[0].validator_logs
    failures = [l for l in logs if type(l.validation_result).__name__ == 'FailResult']
    print(f'Total failures collected: {len(failures)}')
    for f in failures:
        print(f'  - {f.validator_name}')

In [ ]:
# Example 19: Dynamic pipeline construction from config dict
from guardrails.hub import ValidLength, ToxicLanguage, ProfanityFree

VALIDATOR_MAP = {
    'valid_length': lambda cfg: ValidLength(min=cfg.get('min', 5), max=cfg.get('max', 500), on_fail=OnFailAction.EXCEPTION),
    'toxic_language': lambda cfg: ToxicLanguage(on_fail=OnFailAction.EXCEPTION),
    'profanity_free': lambda cfg: ProfanityFree(on_fail=OnFailAction.EXCEPTION),
}

config = {
    'validators': [
        {'name': 'valid_length', 'min': 20, 'max': 1000},
        {'name': 'toxic_language'},
    ]
}

guard = Guard()
for v_cfg in config['validators']:
    builder = VALIDATOR_MAP[v_cfg['name']]
    guard = guard.use(builder(v_cfg))

outcome = guard.validate('This is a dynamically configured validation pipeline for content safety.')
print('dynamic pipeline passed:', outcome.validation_passed)

In [ ]:
# Example 20: Pipeline with metadata shared across all validators
from guardrails.hub import ValidLength

guard = Guard().use(ValidLength(min=10, max=1000, on_fail=OnFailAction.NOOP))
shared_metadata = {
    'user_id': 'u-42',
    'session_id': 'sess-abc123',
    'sources': ['Python documentation says...']
}
outcome = guard.validate(
    'Python is a high-level programming language with clear syntax.',
    metadata=shared_metadata
)
print('shared metadata passed through:', outcome.validation_passed)

In [ ]:
# Example 21: Pipeline benchmarking — timing validator execution
from guardrails.hub import ValidLength, ToxicLanguage, ProfanityFree
guard = (
    Guard()
    .use(ValidLength(min=5, max=500, on_fail=OnFailAction.NOOP))
    .use(ToxicLanguage(on_fail=OnFailAction.NOOP))
    .use(ProfanityFree(on_fail=OnFailAction.NOOP))
)
text = 'This is a benchmarking test for the validation pipeline execution time.'
start = time.perf_counter()
for _ in range(5):
    guard.validate(text)
elapsed = time.perf_counter() - start
print(f'5 validations took {elapsed:.3f}s ({elapsed/5*1000:.1f}ms each)')

## Examples 22–28: Async, Custom + Hub, and Deduplication

In [ ]:
# Example 22: Custom validator mixed with Hub validators in same pipeline
from guardrails.hub import ValidLength
from guardrails.validator_base import Validator, register_validator, PassResult, FailResult

@register_validator(name='ends-with-dot', data_type='string')
class EndsWithDot(Validator):
    def validate(self, value, metadata):
        return PassResult() if value.rstrip().endswith('.') else FailResult('Must end with period')

guard = (
    Guard()
    .use(ValidLength(min=10, max=500, on_fail=OnFailAction.EXCEPTION))
    .use(EndsWithDot(on_fail=OnFailAction.FIX))
)
outcome = guard.validate('The sky is blue')
print('custom + hub combined:', outcome.validated_output)

In [ ]:
# Example 23: Async pipeline — AsyncGuard with multiple validators
from guardrails.hub import ValidLength, ToxicLanguage

async def run_async_pipeline():
    guard = (
        AsyncGuard()
        .use(ValidLength(min=10, max=1000, on_fail=OnFailAction.EXCEPTION))
        .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
    )
    async_oai = openai.AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    outcome = await guard(
        async_oai.chat.completions.create,
        prompt='Describe the internet in one sentence.',
        model=MODEL
    )
    return outcome.validated_output

result = asyncio.run(run_async_pipeline())
print('async pipeline result:', result[:100] if result else 'None')

In [ ]:
# Example 24: Input-only pipeline — all validators on user input before LLM
from guardrails.hub import ValidLength, DetectPII

def validate_and_call(user_msg: str) -> str:
    input_guard = (
        Guard()
        .use(ValidLength(min=5, max=500, on_fail=OnFailAction.EXCEPTION))
        .use(DetectPII(on_fail=OnFailAction.EXCEPTION))
    )
    input_guard.validate(user_msg)
    resp = oai.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': user_msg}]
    )
    return resp.choices[0].message.content

answer = validate_and_call('What is the difference between SQL and NoSQL databases?')
print('input-only pipeline result:', answer[:100])

In [ ]:
# Example 25: Output-only pipeline — all validators on LLM response
from guardrails.hub import ToxicLanguage, ValidLength, ProfanityFree

output_guard = (
    Guard()
    .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
    .use(ValidLength(min=20, max=2000, on_fail=OnFailAction.EXCEPTION))
    .use(ProfanityFree(on_fail=OnFailAction.EXCEPTION))
)
outcome = output_guard(
    oai.chat.completions.create,
    prompt='Explain version control systems briefly.',
    model=MODEL
)
print('output-only pipeline passed:', outcome.validation_passed)

In [ ]:
# Example 26: Global fail action override — set on_fail for entire pipeline
from guardrails.hub import ValidLength, ToxicLanguage

# Apply NOOP globally by setting it on each validator
validators = [
    ValidLength(min=50, max=500, on_fail=OnFailAction.NOOP),
    ToxicLanguage(on_fail=OnFailAction.NOOP),
]
guard = Guard().use_many(*validators)
outcome = guard.validate('Hi!')  # fails both, but NOOP means no exception
print('global NOOP — no exception raised, passed:', outcome.validation_passed)

In [ ]:
# Example 27: Validator deduplication — adding same validator twice
from guardrails.hub import ValidLength
v1 = ValidLength(min=5, max=500, on_fail=OnFailAction.NOOP)
v2 = ValidLength(min=5, max=500, on_fail=OnFailAction.NOOP)
guard = Guard().use(v1).use(v2)  # same validator type added twice
outcome = guard.validate('Some text here.')
print('deduplication test — validators:', len(guard.validators) if hasattr(guard, 'validators') else 'N/A')
print('still validates correctly:', outcome.validation_passed)

In [ ]:
# Example 28: Batch processing — same guard validates multiple texts
from guardrails.hub import ToxicLanguage, ValidLength
guard = (
    Guard()
    .use(ValidLength(min=10, max=500, on_fail=OnFailAction.NOOP))
    .use(ToxicLanguage(on_fail=OnFailAction.NOOP))
)
texts = [
    'A well-written professional response.',
    'Short.',
    'I hate everyone and everything is terrible!',
    'Machine learning powers many modern applications today.',
]
for i, text in enumerate(texts):
    outcome = guard.validate(text)
    print(f'  [{i+1}] passed={outcome.validation_passed}: {text[:40]}')

## Examples 29–32: End-to-End Domain Pipelines

In [ ]:
# Example 29: Inspect guard.validators list
from guardrails.hub import ValidLength, ToxicLanguage, ProfanityFree
guard = (
    Guard()
    .use(ValidLength(min=5, max=500, on_fail=OnFailAction.EXCEPTION))
    .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
    .use(ProfanityFree(on_fail=OnFailAction.EXCEPTION))
)
print('Number of validators:', len(guard.validators))
for v in guard.validators:
    print(f'  - {v.__class__.__name__}')

In [ ]:
# Example 30: Parallel async validation of multiple outputs
from guardrails.hub import ToxicLanguage

async def validate_batch_async(texts):
    guard = AsyncGuard().use(ToxicLanguage(on_fail=OnFailAction.NOOP))
    tasks = [guard.async_validate(t) for t in texts]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    return results

batch = [
    'This is a professional message.',
    'Another helpful response about technology.',
    'Informative content about programming best practices.',
]
results = asyncio.run(validate_batch_async(batch))
for i, r in enumerate(results):
    print(f'  [{i+1}] passed={r.validation_passed if not isinstance(r, Exception) else "ERROR"}')

In [ ]:
# Example 31: Healthcare content pipeline — ToxicLanguage + ValidLength + ContainsString
from guardrails.hub import ToxicLanguage, ValidLength, ContainsString
healthcare_guard = (
    Guard()
    .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
    .use(ValidLength(min=50, max=2000, on_fail=OnFailAction.EXCEPTION))
    .use(ContainsString(search_string='consult', on_fail=OnFailAction.NOOP))  # prefer 'consult a doctor'
)
outcome = healthcare_guard(
    oai.chat.completions.create,
    prompt='What are common symptoms of dehydration? Always recommend consulting a doctor.',
    model=MODEL
)
print('healthcare pipeline passed:', outcome.validation_passed)

In [ ]:
# Example 32: Financial chatbot complete pipeline
# ToxicLanguage + ProfanityFree + ValidLength (content safety + quality)
from guardrails.hub import ToxicLanguage, ProfanityFree, ValidLength

financial_guard = (
    Guard()
    .use(ToxicLanguage(on_fail=OnFailAction.EXCEPTION))
    .use(ProfanityFree(on_fail=OnFailAction.EXCEPTION))
    .use(ValidLength(min=50, max=2000, on_fail=OnFailAction.EXCEPTION))
)
outcome = financial_guard(
    oai.chat.completions.create,
    prompt=(
        'Explain what a mutual fund is to a beginner. '
        'Be professional and include appropriate disclaimers. '
        'Do not provide specific investment advice.'
    ),
    model=MODEL
)
print('financial chatbot pipeline passed:', outcome.validation_passed)
print('response preview:', outcome.validated_output[:150] if outcome.validated_output else 'None')